# RealEstatePRO

In [70]:
#import all necessary libraries
import pandas as pd
import os
from getpass import getpass
from langchain_openai import OpenAIEmbeddings
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import tqdm

In [71]:
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY") or \
    getpass("Enter LangSmith API Key: ")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "aurelioai-langchain-course-agent-executor-openai"

In [68]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") \
    or getpass("Enter your OpenAI API key: ")

## DataFrame import

In [14]:
df = pd.read_csv('NY-House-Dataset.csv')

In [15]:
df.head(10)

,BROKERTITLE,TYPE,PRICE,BEDS,BATH,PROPERTYSQFT,ADDRESS,STATE,MAIN_ADDRESS,ADMINISTRATIVE_AREA_LEVEL_2,LOCALITY,SUBLOCALITY,STREET_NAME,LONG_NAME,FORMATTED_ADDRESS,LATITUDE,LONGITUDE
0,Brokered by Douglas Elliman -111 Fifth Ave,Condo for sale,315000,2,2.000000,1400.000000,2 E 55th St Unit 803,"New York, NY 10022","2 E 55th St Unit 803New York, NY 10022",New York County,New York,Manhattan,East 55th Street,Regis Residence,"Regis Residence, 2 E 55th St #803, New York, N...",40.761255,-73.974483
1,Brokered by Serhant,Condo for sale,195000000,7,10.000000,17545.000000,Central Park Tower Penthouse-217 W 57th New Yo...,"New York, NY 10019",Central Park Tower Penthouse-217 W 57th New Yo...,United States,New York,New York County,New York,West 57th Street,"217 W 57th St, New York, NY 10019, USA",40.766393,-73.980991
2,Brokered by Sowae Corp,House for sale,260000,4,2.000000,2015.000000,620 Sinclair Ave,"Staten Island, NY 10312","620 Sinclair AveStaten Island, NY 10312",United States,New York,Richmond County,Staten Island,Sinclair Avenue,"620 Sinclair Ave, Staten Island, NY 10312, USA",40.541805,-74.196109
3,Brokered by COMPASS,Condo for sale,69000,3,1.000000,445.000000,2 E 55th St Unit 908W33,"Manhattan, NY 10022","2 E 55th St Unit 908W33Manhattan, NY 10022",United States,New York,New York County,New York,East 55th Street,"2 E 55th St, New York, NY 10022, USA",40.761398,-73.974613
4,Brokered by Sotheby's International Realty - E...,Townhouse for sale,55000000,7,2.373861,14175.000000,5 E 64th St,"New York, NY 10065","5 E 64th StNew York, NY 10065",United States,New York,New York County,New York,East 64th Street,"5 E 64th St, New York, NY 10065, USA",40.767224,-73.969856
5,Brokered by Sowae Corp,House for sale,690000,5,2.000000,4004.000000,584 Park Pl,"Brooklyn, NY 11238","584 Park PlBrooklyn, NY 11238",United States,New York,Kings County,Brooklyn,Park Place,"584 Park Pl, Brooklyn, NY 11238, USA",40.674363,-73.958725
6,Brokered by Douglas Elliman - 575 Madison Ave,Condo for sale,899500,2,2.000000,2184.207862,157 W 126th St Unit 1B,"New York, NY 10027","157 W 126th St Unit 1BNew York, NY 10027",New York,New York County,New York,Manhattan,157,"157 W 126th St #1b, New York, NY 10027, USA",40.809448,-73.946777
7,Brokered by Connie Profaci Realty,House for sale,16800000,8,16.000000,33000.000000,177 Benedict Rd,"Staten Island, NY 10304","177 Benedict RdStaten Island, NY 10304",United States,New York,Richmond County,Staten Island,Benedict Road,"177 Benedict Rd, Staten Island, NY 10304, USA",40.595002,-74.106424
8,Brokered by Pantiga Group Inc.,Co-op for sale,265000,1,1.000000,750.000000,875 Morrison Ave Apt 3M,"Bronx, NY 10473","875 Morrison Ave Apt 3MBronx, NY 10473",Bronx County,The Bronx,East Bronx,Morrison Avenue,Parking lot,"Parking lot, 875 Morrison Ave #3m, Bronx, NY 1...",40.821586,-73.874089
9,Brokered by CENTURY 21 MK Realty,Co-op for sale,440000,2,1.000000,978.000000,1350 Ocean Pkwy Apt 5G,"Brooklyn, NY 11230","1350 Ocean Pkwy Apt 5GBrooklyn, NY 11230",New York,Kings County,Brooklyn,Midwood,1350,"1350 Ocean Pkwy #5g, Brooklyn, NY 11230, USA",40.615738,-73.969694


In [27]:
df.describe()

,price,beds,bath,propertysqft,latitude,longitude
count,4.801000e+03,4801.000000,4801.000000,4801.000000,4801.000000,4801.000000
mean,2.356940e+06,3.356801,2.373861,2184.207862,40.714227,-73.941601
std,3.135525e+07,2.602315,1.946962,2377.140894,0.087676,0.101082
min,2.494000e+03,1.000000,0.000000,230.000000,40.499546,-74.253033
25%,4.990000e+05,2.000000,1.000000,1200.000000,40.639375,-73.987143
50%,8.250000e+05,3.000000,2.000000,2184.207862,40.726749,-73.949189
75%,1.495000e+06,4.000000,3.000000,2184.207862,40.771923,-73.870638
max,2.147484e+09,50.000000,50.000000,65535.000000,40.912729,-73.702450


In [19]:
df.shape

(4801, 17)

In [20]:
#make all columns lowercase
df.columns = [col.lower() for col in df.columns]

In [28]:
df.columns

Index(['brokertitle', 'type', 'price', 'beds', 'bath', 'propertysqft',
       'address', 'state', 'main_address', 'administrative_area_level_2',
       'locality', 'sublocality', 'street_name', 'long_name',
       'formatted_address', 'latitude', 'longitude'],
      dtype='object')

### Save embeddings in FAISS

What is FAISS: TODO

In [ ]:
index_path = "faiss_index_dir"
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
if os.path.exists(index_path):
    # Load existing FAISS index
    vector_store = FAISS.load_local(index_path, embeddings)
    print("Loaded existing FAISS index.")

In [48]:
# Create a concise textual representation for embedding
df['text_to_embed'] = (
    df['brokertitle'].astype(str) + ", " +
    df['type'].astype(str) + ", Price: " + df['price'].astype(str) + "$, " +
    "Beds: " + df['beds'].astype(str) + ", Baths: " + df['bath'].astype(str) + ", " +
    "Size: " + df['propertysqft'].astype(str) + " sqft, " +
    "Address: " + df['address'].astype(str) + ", " +
    "Locality: " + df['locality'].astype(str) + ", " +
    "State: " + df['state'].astype(str)
)

In [49]:
df

,brokertitle,type,price,beds,bath,propertysqft,address,state,main_address,administrative_area_level_2,locality,sublocality,street_name,long_name,formatted_address,latitude,longitude,text_to_embed
0,Brokered by Douglas Elliman -111 Fifth Ave,Condo for sale,315000,2,2.000000,1400.000000,2 E 55th St Unit 803,"New York, NY 10022","2 E 55th St Unit 803New York, NY 10022",New York County,New York,Manhattan,East 55th Street,Regis Residence,"Regis Residence, 2 E 55th St #803, New York, N...",40.761255,-73.974483,"Brokered by Douglas Elliman -111 Fifth Ave, C..."
1,Brokered by Serhant,Condo for sale,195000000,7,10.000000,17545.000000,Central Park Tower Penthouse-217 W 57th New Yo...,"New York, NY 10019",Central Park Tower Penthouse-217 W 57th New Yo...,United States,New York,New York County,New York,West 57th Street,"217 W 57th St, New York, NY 10019, USA",40.766393,-73.980991,"Brokered by Serhant, Condo for sale, Price: 19..."
2,Brokered by Sowae Corp,House for sale,260000,4,2.000000,2015.000000,620 Sinclair Ave,"Staten Island, NY 10312","620 Sinclair AveStaten Island, NY 10312",United States,New York,Richmond County,Staten Island,Sinclair Avenue,"620 Sinclair Ave, Staten Island, NY 10312, USA",40.541805,-74.196109,"Brokered by Sowae Corp, House for sale, Price:..."
3,Brokered by COMPASS,Condo for sale,69000,3,1.000000,445.000000,2 E 55th St Unit 908W33,"Manhattan, NY 10022","2 E 55th St Unit 908W33Manhattan, NY 10022",United States,New York,New York County,New York,East 55th Street,"2 E 55th St, New York, NY 10022, USA",40.761398,-73.974613,"Brokered by COMPASS, Condo for sale, Price: 69..."
4,Brokered by Sotheby's International Realty - E...,Townhouse for sale,55000000,7,2.373861,14175.000000,5 E 64th St,"New York, NY 10065","5 E 64th StNew York, NY 10065",United States,New York,New York County,New York,East 64th Street,"5 E 64th St, New York, NY 10065, USA",40.767224,-73.969856,Brokered by Sotheby's International Realty - E...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4796,Brokered by COMPASS,Co-op for sale,599000,1,1.000000,2184.207862,222 E 80th St Apt 3A,"Manhattan, NY 10075","222 E 80th St Apt 3AManhattan, NY 10075",New York,New York County,New York,Manhattan,222,"222 E 80th St #3a, New York, NY 10075, USA",40.774350,-73.955879,"Brokered by COMPASS, Co-op for sale, Price: 59..."
4797,Brokered by Mjr Real Estate Llc,Co-op for sale,245000,1,1.000000,2184.207862,97-40 62 Dr Unit Lg,"Rego Park, NY 11374","97-40 62 Dr Unit LgRego Park, NY 11374",United States,New York,Queens County,Queens,62nd Drive,"97-40 62nd Dr, Rego Park, NY 11374, USA",40.732538,-73.860152,"Brokered by Mjr Real Estate Llc, Co-op for sal..."
4798,Brokered by Douglas Elliman - 575 Madison Ave,Co-op for sale,1275000,1,1.000000,2184.207862,427 W 21st St Unit Garden,"New York, NY 10011","427 W 21st St Unit GardenNew York, NY 10011",United States,New York,New York County,New York,West 21st Street,"427 W 21st St, New York, NY 10011, USA",40.745882,-74.003398,"Brokered by Douglas Elliman - 575 Madison Ave,..."
4799,Brokered by E Realty International Corp,Condo for sale,598125,2,1.000000,655.000000,91-23 Corona Ave Unit 4G,"Elmhurst, NY 11373","91-23 Corona Ave Unit 4GElmhurst, NY 11373",New York,Queens County,Queens,Flushing,91-23,"91-23 Corona Ave. #4b, Flushing, NY 11373, USA",40.742770,-73.872752,"Brokered by E Realty International Corp, Condo..."


In [50]:
# Create LangChain Documents, saving only essential metadata
documents = []
for idx, row in df.iterrows():
    metadata = {
        "beds": row['beds'],
        "bath": row['bath'],
        "address": row['address'],
        "state": row['state'],
        "propertysqft": row['propertysqft'],
        "price": row['price'],
    }
    doc = Document(page_content=row['text_to_embed'], metadata=metadata)
    documents.append(doc)

In [51]:
documents[0]  # Display the first document to verify

Document(metadata={'beds': 2, 'bath': 2.0, 'address': '2 E 55th St Unit 803', 'state': 'New York, NY 10022', 'propertysqft': 1400.0, 'price': 315000}, page_content='Brokered by Douglas Elliman  -111 Fifth Ave, Condo for sale, Price: 315000$, Beds: 2, Baths: 2.0, Size: 1400.0 sqft, Address: 2 E 55th St Unit 803, Locality: New York, State: New York, NY 10022')

In [52]:
embedding_dim = len(embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [ ]:
# Add documents to the vector store 
vector_store.add_documents(documents)


  0%|          | 0/4801 [00:00<?, ?it/s]

100%|██████████| 4801/4801 [00:16<00:00, 293.76it/s]


In [61]:
results = vector_store.similarity_search(
    "House in New York with 3 bedrooms and not more than 2 bathrooms between 1000000$ and 3000000$",
    k=2
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Brokered by COMPASS, House for sale, Price: 14000000$, Beds: 3, Baths: 2.3738608579684373, Size: 23027.0 sqft, Address: 39 Eldridge St, Locality: New York, State: Manhattan, NY 10002 [{'beds': 3, 'bath': 2.3738608579684373, 'address': '39 Eldridge St', 'state': 'Manhattan, NY 10002', 'propertysqft': 23027.0, 'price': 14000000}]
* Brokered by Carina Realty Inc, House for sale, Price: 1350000$, Beds: 3, Baths: 2.3738608579684373, Size: 2184.207862 sqft, Address: 219 E 115th St, Locality: New York, State: Nyc, NY 10029 [{'beds': 3, 'bath': 2.3738608579684373, 'address': '219 E 115th St', 'state': 'Nyc, NY 10029', 'propertysqft': 2184.207862, 'price': 1350000}]


In [66]:
vector_store.save_local("faiss_index_dir")

## LangChain

### Tools

In [75]:
from langchain_core.tools import tool

@tool
def retrieve(query: str) -> str:
    """Retrieve relevant real estate listings based on the query."""
    results = vector_store.similarity_search(query, k=5)
    if not results:
        return "No relevant listings found."
    
    response = "Here are some relevant listings:\n"
    for i, res in enumerate(results, 1):
        response += (f"{i}. {res.page_content} | "
                     f"Beds: {res.metadata.get('beds')}, "
                     f"Baths: {res.metadata.get('bath')}, "
                     f"Price: {res.metadata.get('price')}$, "
                     f"Address: {res.metadata.get('address')}, "
                     f"State: {res.metadata.get('state')}, "
                     f"Size: {res.metadata.get('propertysqft')} sqft\n")
    return response



In [76]:
retrieve

StructuredTool(name='retrieve', description='Retrieve relevant real estate listings based on the query.', args_schema=<class 'langchain_core.utils.pydantic.retrieve'>, func=<function retrieve at 0x0000029408B8E8E0>)

In [77]:
print(f"{retrieve.name=}\n{retrieve.description=}")

retrieve.name='retrieve'
retrieve.description='Retrieve relevant real estate listings based on the query.'


In [78]:
retrieve.args_schema.model_json_schema()

{'description': 'Retrieve relevant real estate listings based on the query.',
 'properties': {'query': {'title': 'Query', 'type': 'string'}},
 'required': ['query'],
 'title': 'retrieve',
 'type': 'object'}

### Creating Agent

In [72]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You're a helpful assistant that will provide information about real estate listings and "
        "assist users in finding the right property suggesting listings based on their preferences and requirements. "
        "When answering a user's question "
        "you should first use one of the tools provided. After using a "
        "tool the tool output will be provided in the "
        "'scratchpad' below. If you have an answer in the "
        "scratchpad you should not use any more tools and "
        "instead answer directly to the user."
    )),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [74]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0.0,
)

In [80]:
from langchain_core.runnables.base import RunnableSerializable

tools = [retrieve]

# define the agent runnable
agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
    }
    | prompt
    | llm.bind_tools(tools, tool_choice="any")
)

In [81]:
tool_call = agent.invoke({"input": "I want to buy a house in Manhattan, with 3 bedrooms and 2 bathrooms. I have a budget of $1 million.", "chat_history": []})
tool_call

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_mnfL18hQSZoehxq359VzQaj6', 'function': {'arguments': '{"query":"Manhattan house for sale 3 bedrooms 2 bathrooms under $1 million"}', 'name': 'retrieve'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 159, 'total_tokens': 186, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CPBsBeevcPUrwUzHA7XlrdrXIqhIS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--6ef2c989-406e-4aa5-a566-b86aa949d1df-0', tool_calls=[{'name': 'retrieve', 'args': {'query': 'Manhattan house for sale 3 bedrooms 2 bathrooms under $1 million'}, 'id': 'call_mnfL18hQSZoehxq359VzQaj6', 'type': 'tool_c

In [82]:
tool_call.tool_calls

[{'name': 'retrieve',
  'args': {'query': 'Manhattan house for sale 3 bedrooms 2 bathrooms under $1 million'},
  'id': 'call_mnfL18hQSZoehxq359VzQaj6',
  'type': 'tool_call'}]

In [83]:
# create tool name to function mapping
name2tool = {tool.name: tool.func for tool in tools}

In [84]:
tool_exec_content = name2tool[tool_call.tool_calls[0]["name"]](
    **tool_call.tool_calls[0]["args"]
)
tool_exec_content

"Here are some relevant listings:\n1. Brokered by Sotheby's International Realty - East Side Manhattan Brokerage, For sale, Price: 899000$, Beds: 3, Baths: 1.0, Size: 1100.0 sqft, Address: 529 W 42nd St Apt 2G, Locality: New York County, State: New York, NY 10036 | Beds: 3, Baths: 1.0, Price: 899000$, Address: 529 W 42nd St Apt 2G, State: New York, NY 10036, Size: 1100.0 sqft\n2. Brokered by COMPASS, Condo for sale, Price: 2250000$, Beds: 3, Baths: 1.0, Size: 1710.0 sqft, Address: 181 Hudson St Ste 2A, Locality: New York County, State: Manhattan, NY 10013 | Beds: 3, Baths: 1.0, Price: 2250000$, Address: 181 Hudson St Ste 2A, State: Manhattan, NY 10013, Size: 1710.0 sqft\n3. Brokered by COMPASS, Condo for sale, Price: 999000$, Beds: 3, Baths: 1.0, Size: 600.0 sqft, Address: 10 W End Ave Apt 8J, Locality: New York County, State: Manhattan, NY 10023 | Beds: 3, Baths: 1.0, Price: 999000$, Address: 10 W End Ave Apt 8J, State: Manhattan, NY 10023, Size: 600.0 sqft\n4. Brokered by Keller Will